# Benchmarks CPU vs GPU (NumPy, cuDF/cuML, JAX) — Base


Comparamos tiempos de operaciones matriciales y ML entre CPU y GPU cuando el runtime lo permite.

Los checks condicionales facilitan el aprendizaje y la depuración en CPU. Sin embargo, una ejecución sin backend GPU **no sirve como evidencia evaluable del benchmark GPU**. Antes de medir, verifica y registra el dispositivo real.

In [ ]:
import time
import numpy as np

N = 2000
A = np.random.rand(N, N).astype("float32")
_ = A @ A  # warmup CPU fuera de la medición
t0 = time.perf_counter()
_ = A @ A
t_cpu = time.perf_counter() - t0
print("Prod. matricial NumPy (CPU):", round(t_cpu, 3), "s")


In [ ]:
# JAX: warmup y sincronización obligatorios para medir trabajo real.
try:
    import jax
    import jax.numpy as jnp

    HAS_JAX = True
    JAX_GPU_AVAILABLE = any(device.platform == "gpu" for device in jax.devices())
    B = jnp.array(A)
    jax_matmul = jax.jit(lambda x: x @ x)

    jax_matmul(B).block_until_ready()  # compila y termina el warmup antes de medir
    t0 = time.perf_counter()
    result_jax = jax_matmul(B)
    result_jax.block_until_ready()  # espera a la GPU/CPU antes de detener el tiempo
    t_jax = time.perf_counter() - t0

    print("JAX", jax.__version__, "| Backend:", jax.default_backend())
    print("Dispositivos:", jax.devices())
    print("Prod. matricial JAX:", round(t_jax, 3), "s")
    if not JAX_GPU_AVAILABLE:
        print("⚠️ Resultado JAX en CPU: demostración válida, NO evidencia GPU evaluable.")
except Exception as e:
    HAS_JAX = False
    JAX_GPU_AVAILABLE = False
    print("JAX no disponible:", e)


In [ ]:
# cuDF + cuML: esta ruta solo acredita GPU si RAPIDS usa CUDA correctamente.
try:
    import cupy as cp
    import cudf
    import cuml
    from cuml.linear_model import LogisticRegression as CuLogReg
    import pandas as pd

    HAS_RAPIDS = True
    df = pd.DataFrame({
        "x1": np.random.rand(5000),
        "x2": np.random.rand(5000),
        "y": np.random.randint(0, 2, 5000),
    })
    gdf = cudf.DataFrame.from_pandas(df)
    X = gdf[["x1", "x2"]].astype("float32")
    y = gdf["y"].astype("int32")

    CuLogReg(max_iter=50).fit(X.iloc[:1000], y.iloc[:1000])
    cp.cuda.Stream.null.synchronize()  # warmup terminado
    t0 = time.perf_counter()
    model_gpu = CuLogReg(max_iter=200).fit(X, y)
    cp.cuda.Stream.null.synchronize()  # trabajo GPU terminado antes de cerrar el tiempo
    t_fit = time.perf_counter() - t0

    device_name = cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)["name"]
    if isinstance(device_name, bytes):
        device_name = device_name.decode()
    print("cuDF", cudf.__version__, "| cuML", cuml.__version__)
    print("Dispositivo NVIDIA:", device_name)
    print("cuML LogisticRegression fit:", round(t_fit, 3), "s")
except Exception as e:
    HAS_RAPIDS = False
    print("RAPIDS GPU no disponible:", e)
    print("⚠️ Puedes revisar el notebook en CPU, pero NO generar evidencia GPU evaluable.")


### Ejercicios

1. Aumenta `N` y observa la evolución de tiempos.
2. Compara `jax.jit` con la versión sin compilar, siempre con warmup y `.block_until_ready()`.
3. Usa cuML para otro clasificador y cronometra tras sincronizar el stream GPU.
4. Registra GPU, versiones, backend y dispositivo junto a los tiempos.

Los tiempos obtenidos únicamente en CPU pueden usarse para depurar o discutir el método, pero no acreditan el benchmark GPU evaluable.